# Galyleo チュートリアル　1
### このJupyter notebookで、世界の天気をNOAA（アメリカ海洋大気庁）よりデータ抽出し、データ集約するデータプレーン「Galyleo Setvice」へアップロードします。
<img src="attachment:b0f0524a-5600-4a95-a571-023453409c23.png" width="600px">

In [ ]:
# モジュール読み込み
import os
import requests
import pandas as pd
from sdtp import RowTable

### 1. Galyleo Service用のパス＆認証を指定します。

In [ ]:
PUBLISH_URL = "http://galyleo-service.jhub-kct-free.svc.cluster.local:5000/services/galyleo/publish_data"
API_TOKEN = os.getenv('JUPYTERHUB_API_TOKEN')

### 2. NOAA用都市IDを指定します。

In [ ]:
STATIONS = {
    "London": "03772099999",
    "Tokyo": "47662099999",
    "Beijing": "54511099999",
    "Sydney": "94767099999",
    "Rio": "83746099999"
}

### 3. 選択した都市の NOAA気候データを読み取ります。

In [ ]:
all_data = []
for city, sid in STATIONS.items():
    url = f"https://www.ncei.noaa.gov/data/global-summary-of-the-day/access/2023/{sid}.csv"
    df = pd.read_csv(url)[['DATE', 'TEMP', 'PRCP']]
    df['CITY'] = city
    all_data.append(df)

In [ ]:
final_df = pd.concat(all_data)
print(final_df)

### 4. SDMLスキーマ - Galyleo Service用に明示的にカラムの型指定をします。
※日付はdate型、数値はnumber型、文字はstring型になります。

In [ ]:
schema = [
    {"name": "DATE", "type": "date"},
    {"name": "TEMP", "type": "number"},
    {"name": "PRCP", "type": "number"},
    {"name": "CITY", "type": "string"}
]

### 5. SDMLスキーマ準拠にしたデータフレームを、Galyleo Serviceに送ります。

In [ ]:
table = RowTable(schema, final_df.values.tolist())
headers = {'Authorization': f'token {API_TOKEN}'}
r = requests.post(PUBLISH_URL, headers=headers, json={
    "table": table.to_dictionary(),
    "name": "global_weather_tutorial.sdml"
})

In [ ]:
print(f"Status: {r.status_code} - Global Weather Published!" if r.ok else f"Failed: {r.text}")
# "Status 200" が返ってくればOK.

### Galyleo Serviceを開いて、データフレームがアップロードされていることを確認しましょう。
<img src="attachment:99a43819-c112-45fb-ae30-c37a2c32933e.png" width="280px">  

# ↓
<img src="attachment:b3e8bea6-cdb8-480b-ac31-d19c406a6f3e.png" width="400px">  

# ↓
<img src="attachment:fc5492c4-9bdc-4f73-9e50-1cf3129ad156.png" width="400px">  